In [1]:
# Install the Hugging Face Datasets library used to load the source datasets.
!pip install -q datasets

In [ ]:
# Load the iamtangsang Nepali-English translation dataset.
from datasets import load_dataset, concatenate_datasets

iamts = load_dataset("iamTangsang/Nepali-to-English-Translation-Dataset")
print(iamts)

In [3]:
# Merge the train, validation, and test splits so they can be processed together.
iamtangsang = concatenate_datasets([
    iamts["train"],
    iamts["validation"],
    iamts["test"]
])

In [ ]:
# Inspect the available columns, dataset size, and overall structure.
print(iamtangsang.column_names)
print(iamtangsang)
print(iamtangsang[0])

In [ ]:
# Load the HIMALAYA Nepali SFT dataset containing conversational examples.
himalaya = load_dataset("himalaya-ai/nepali-sft-dataset")
print(himalaya)

In [ ]:
print(himalaya["train"][0])

#### Extracting translation pairs from Himalaya-sft dataset


In [7]:
# retrieve messages by their conversation role.

def get_message(example, role):
    for message in example["conversations"]:
        if message["from"] == role:
            return message["value"]
    return ""

def get_human(example):
    return get_message(example, "human")

def get_gpt(example):
    return get_message(example, "gpt")


In [ ]:
# Identify HIMALAYA examples whose human message contains a translation instruction.

import re

translation_pattern = re.compile(
    r"^Translate the following (?:Nepali text to English|English text to Nepali)\s*:?\s*"
)

def has_translation_pattern(example):
    return bool(translation_pattern.search(get_human(example)))

translation_candidates = himalaya["train"].filter(
    has_translation_pattern,
    desc="Searching for translation samples",
)

print("Total translation candidates:", len(translation_candidates))


In [9]:
# Patterns used to identify the translation direction from the instruction.
eng_to_nep_pattern = re.compile(
    r"translate\s+the\s+following\s+english\s+text\s+to\s+nepali",
    re.IGNORECASE,
)

nep_to_eng_pattern = re.compile(
    r"translate\s+the\s+following\s+nepali\s+text\s+to\s+english",
    re.IGNORECASE,
)

In [10]:

def classify_translation(example):
    human = get_human(example)

    if eng_to_nep_pattern.search(human):
        return "eng_to_nep"
    if nep_to_eng_pattern.search(human):
        return "nep_to_eng"
    return "unknown"

In [ ]:
# Convert the selected HIMALAYA examples into a simple tabular structure.
import pandas as pd

results = []

for row in translation_candidates:
    results.append({
        "human": get_human(row),
        "gpt": get_gpt(row),
        "direction": classify_translation(row),
    })

translation_df = pd.DataFrame(results)

print("Direction counts:")
display(translation_df["direction"].value_counts().rename_axis("direction").to_frame("count"))

In [ ]:
eng_to_nep = translation_df[translation_df["direction"] == "eng_to_nep"]
nep_to_eng = translation_df[translation_df["direction"] == "nep_to_eng"]

display(eng_to_nep.sample(min(10, len(eng_to_nep)), random_state=42))
display(nep_to_eng.sample(min(10, len(nep_to_eng)), random_state=42))

Remove the translation instruction from the human column and keep only the actual text to be translated.

In [13]:
# Remove the instruction prefix and keep only the text that should be translated.
def extract_text(human_text):
    if pd.isna(human_text):
        return ""

    text = str(human_text).strip()

    # Remove translation instruction
    patterns = [
        r"^Translate the following Nepali text to English\s*:?\s*",
        r"^Translate the following English text to Nepali\s*:?\s*",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    return text.strip()


In [14]:
# Apply instruction removal to every Himalaya human message.
translation_df["human"] = translation_df["human"].apply(extract_text)

In [ ]:
display(
    translation_df[
        ["human", "gpt", "direction"]
    ].sample(20, random_state=42)
)


Standardize iamtangsang

In [16]:
# Convert iamtangsang to pandas and standardize its columns and metadata.
iam_df = iamtangsang.to_pandas()

iam_df = iam_df[["source", "target"]].copy()

iam_df["direction"] = "nep_to_eng"
iam_df["dataset"] = "iamTangsang"

Standardize himalaya ne-en

In [17]:
# Rename columns to the common schema.
himalaya_ne_en = translation_df[
    translation_df["direction"] == "nep_to_eng"][
    ["human", "gpt"]
].copy()
himalaya_ne_en = himalaya_ne_en.rename(
    columns={
        "human": "source",
        "gpt": "target"
    }
)
himalaya_ne_en["direction"] = "nep_to_eng"
himalaya_ne_en["dataset"] = "himalaya_nep_eng"


Standardize himalaya en-ne

In [18]:
himalaya_en_ne = translation_df[
    translation_df["direction"] == "eng_to_nep"][
    ["human", "gpt"]
].copy()
himalaya_en_ne = himalaya_en_ne.rename(
    columns={
        "human": "source",
        "gpt": "target"
    }
)
himalaya_en_ne["direction"] = "eng_to_nep"
himalaya_en_ne["dataset"] = "himalaya_eng_nep"


#### Combining the three standardized datasets


In [19]:
# Combine all standardized datasets into one DataFrame.
combined_df = pd.concat(
    [
        iam_df,
        himalaya_ne_en,
        himalaya_en_ne
    ],
    ignore_index=True
)

In [ ]:
print("\nFinal shape:", combined_df.shape)

print("\nDataset distribution:")
print(combined_df["dataset"].value_counts())

print("\nDirection distribution:")
print(combined_df["direction"].value_counts())

display(combined_df.head(10))

In [ ]:
# Check for missing values before text cleaning.
print("Missing values:")
print(combined_df[["source", "target", "direction"]].isna().sum())


In [ ]:
# Convert text to strings, trim surrounding whitespace, and check empty pairs.
combined_df["source"] = combined_df["source"].astype(str).str.strip()
combined_df["target"] = combined_df["target"].astype(str).str.strip()

removed_pairs = combined_df[
    (combined_df["source"] == "") |
    (combined_df["target"] == "")
].copy()
print("Removed pairs:", len(removed_pairs))
display(removed_pairs)

In [ ]:
#Remove the exact pairs
combined_df = combined_df.drop(removed_pairs.index)

print("After removing empty pairs:", combined_df.shape)

In [24]:
def normalize_text(text):
    text = re.sub(r"\s+", " ", text) # Collapse repeated whitespace (spaces, tabs, and line breaks) to one space.
    return text.strip() # Remove space from beginning and the end.


combined_df["source"] = combined_df["source"].apply(normalize_text)
combined_df["target"] = combined_df["target"].apply(normalize_text)

In [ ]:
# Inspect exact duplicate source-target pairs before removing them.
duplicates = combined_df[
    combined_df.duplicated(
        subset=["source", "target"],
        keep=False
    )
].sort_values(["source", "target"])

print("Duplicate rows:", len(duplicates))
print("Unique duplicate pairs:",
      duplicates[["source", "target"]].drop_duplicates().shape[0])

display(
    duplicates[
        ["source", "target", "direction", "dataset"]
    ]
)

In [ ]:
#duplicate dataset-count analysis
duplicate_pairs = (
    combined_df
    .groupby(["source", "target"])["dataset"]
    .nunique()
)

cross_dataset_pairs = duplicate_pairs[
    duplicate_pairs > 1
]

print(
    "Pairs appearing in multiple datasets:",
    len(cross_dataset_pairs)
)


In [ ]:
# Keep one copy of each exact source-target pair.
print("Before:", len(combined_df))

cleaned_df = combined_df.drop_duplicates(
    subset=["source", "target"],
    keep="first"
).reset_index(drop=True)

print("After:", len(cleaned_df))
print("Removed:", len(combined_df) - len(cleaned_df))

In [ ]:
# Check conflicting translations same source but different targets or vice versa
source_target_counts = (
    cleaned_df
    .groupby(["source", "direction"])["target"]
    .nunique()
)

conflicting_sources = source_target_counts[
    source_target_counts > 1
]

print(
    "Sources with multiple translations:",
    len(conflicting_sources)
)

# Same target but different sources
target_source_counts = (
    cleaned_df
    .groupby(["target", "direction"])["source"]
    .nunique()
)

conflicting_targets = target_source_counts[
    target_source_counts > 1
]

print(
    "Targets with multiple sources:",
    len(conflicting_targets)
)

In [ ]:
# inspect the sources with different targets
conflicting_sources_df = (
    cleaned_df[
        cleaned_df["source"].isin(
            conflicting_sources.index.get_level_values("source")
        )
    ]
    .sort_values(["source", "direction"])
)
print("Sources with multiple translations:", len(conflicting_sources_df))
display(
    conflicting_sources_df[
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

In [30]:
#Normalize ONLY the conflicting targets for comparison
import unicodedata

def normalize_conflicts(text):
  text = text.lower() # Lowercase

  # Remove Unicode punctuation
  text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith("P")
    )
  # Normalize whitespace
  text = re.sub(r"\s+", " ", text)
  return text.strip()

conflicting_sources_df["target_normalized"] = ( conflicting_sources_df["target"] .apply(normalize_conflicts) )


In [ ]:
# Display the original target alongside the normalized target
display( conflicting_sources_df[ [ "source", "target", "target_normalized", "direction", "dataset" ] ].head(100) )

In [ ]:

# Find rows where the same source has targets that
# become identical after normalization

normalized_duplicates = (
    conflicting_sources_df[
        conflicting_sources_df.duplicated(
            subset=["source", "target_normalized"],
            keep=False
        )
    ]
    .sort_values(["source", "target_normalized"])
)

print("Normalized duplicate rows:", len(normalized_duplicates))

display(
    normalized_duplicates[
        [
            "source",
            "target",
            "target_normalized",
            "direction",
            "dataset"
        ]
    ].head(100)
)



In [ ]:
# Keep the first row for each normalized duplicate pair
rows_to_remove = normalized_duplicates.duplicated(
    subset=["source", "target_normalized"],
    keep="first"
)

# Get the indexes of duplicate rows that should be removed
duplicate_indices = normalized_duplicates.index[rows_to_remove]
display(duplicate_indices)
print("Normalized source-target duplicates to remove:", len(duplicate_indices))

# Inspect rows before removing
display(
    cleaned_df.loc[
        duplicate_indices,
        ["source", "target", "direction", "dataset"]
    ].head(50)
)
# Remove those rows from the original cleaned dataset
cleaned_df = cleaned_df.drop(index=duplicate_indices).copy()

print("After removing normalized duplicates:", cleaned_df.shape)


In [ ]:
#inspect the targets with different sources
conflicting_target_df = (
    cleaned_df[
        cleaned_df["target"].isin(
            conflicting_targets.index.get_level_values("target")
        )
    ]
    .sort_values(["target", "direction"])
)
print("Targets with multiple sources:", len(conflicting_target_df))
display(
    conflicting_target_df[
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

In [35]:
#apply normalization in sources of same target but multiple sources
conflicting_target_df["source_normalized"] = ( conflicting_target_df["source"] .apply(normalize_conflicts) )

In [ ]:
display( conflicting_target_df[ [ "source", "target", "source_normalized", "direction", "dataset" ] ].head(100) )

In [ ]:

# Find rows where the same targets has source that
# become identical after normalization

normalized_duplicates_ts = (
    conflicting_target_df[
        conflicting_target_df.duplicated(
            subset=["source_normalized", "target"],
            keep=False
        )
    ]
    .sort_values(["source_normalized", "target"])
)

print("Normalized duplicate rows:", len(normalized_duplicates_ts))

display(
    normalized_duplicates_ts[
        [
            "source",
            "source_normalized",
            "target",
            "direction",
            "dataset"
        ]
    ].head(100)
)



In [ ]:
rows_to_remove_ts = normalized_duplicates_ts.duplicated(
    subset=["target", "source_normalized"],
    keep="first"
)

duplicate_indices_ts = normalized_duplicates_ts.index[rows_to_remove_ts]

print("Normalized target-source duplicates to remove:", len(duplicate_indices_ts))
display(duplicate_indices_ts)
# Inspect rows before removing
display(
    cleaned_df.loc[
        duplicate_indices_ts,
        ["source", "target", "direction", "dataset"]
    ].head(100)
)

# Remove duplicates
cleaned_df = cleaned_df.drop(index=duplicate_indices_ts).copy()

print("After removing normalized target-source duplicates:", cleaned_df.shape)



In [ ]:
# Re-check dataset and direction distributions after deduplication.
print(cleaned_df["dataset"].value_counts())
print(cleaned_df["direction"].value_counts())

#### Language/Script validation

In [ ]:
import re

def has_devanagari(text):
    return bool(re.search(r'[\u0900-\u097F]', text))

def has_latin(text):
    return bool(re.search(r'[A-Za-z]', text))


# Flag rows where either side contains the wrong script
mixed_script = cleaned_df[
    (
        # Nepali → English:
        # source should NOT contain Latin
        # target should NOT contain Devanagari
        (cleaned_df["direction"] == "nep_to_eng") &
        (
            cleaned_df["source"].apply(has_latin) |
            cleaned_df["target"].apply(has_devanagari)
        )
    )
    |
    (
        # English → Nepali:
        # source should NOT contain Devanagari
        # target should NOT contain Latin
        (cleaned_df["direction"] == "eng_to_nep") &
        (
            cleaned_df["source"].apply(has_devanagari) |
            cleaned_df["target"].apply(has_latin)
        )
    )
]

print("Potentially mixed-script rows:", len(mixed_script))

display(
    mixed_script[
        ["source", "target", "direction", "dataset"]
    ]
)

In [41]:
# Word counts for source and target
cleaned_df["source_word_count"] = cleaned_df["source"].str.split().str.len()
cleaned_df["target_word_count"] = cleaned_df["target"].str.split().str.len()

# Symmetric length ratio:
# always = larger side / smaller side
cleaned_df["length_ratio"] = (
    cleaned_df[["source_word_count", "target_word_count"]].max(axis=1)
    /
    cleaned_df[["source_word_count", "target_word_count"]].min(axis=1)
)



In [ ]:
# Inspect how many examples would be affected by increasingly large length ratios.
for threshold in [5, 10, 20, 50, 100]:
    count = (cleaned_df["length_ratio"] >= threshold).sum()
    print(f"Ratio >= {threshold}: {count} rows")

In [ ]:
extreme_5 = (
    cleaned_df[
        cleaned_df["length_ratio"] >= 5
    ]
    .sort_values("length_ratio", ascending=False)
)

print("Total pairs with ratio >= 5:", len(extreme_5))

display(
    extreme_5[
        [
            "source",
            "target",
            "source_word_count",
            "target_word_count",
            "length_ratio",
            "direction",
            "dataset"
        ]
    ]
)

dropping all length ratio> 5

In [ ]:
# Remove pairs with a source/target word-count ratio of 5 or greater.
cleaned_df = cleaned_df[
    cleaned_df["length_ratio"] < 5
].copy()

print("Remaining rows:", len(cleaned_df))

In [ ]:
# Find pairs where the source and target are exactly identical.
same_text = cleaned_df[
    cleaned_df["source"] == cleaned_df["target"]
]
print("Same source and target:", len(same_text))
same_text[["source", "target", "dataset"]]

In [46]:
# Remove identical source-target pairs because they are not useful translation examples.
cleaned_df = cleaned_df.drop(same_text.index).copy()

In [ ]:
same_text_check = cleaned_df[
    cleaned_df["source"] == cleaned_df["target"]
]

print("Same source and target remaining:", len(same_text_check))

In [48]:
# Create a reversed copy so every valid pair can also be used in the opposite direction.
def reverse_translation(row):
    if row["direction"] == "nep_to_eng":
        new_direction = "eng_to_nep"
    else:
        new_direction = "nep_to_eng"

    return pd.Series({
        "source": row["target"],
        "target": row["source"],
        "direction": new_direction,
        "dataset": row["dataset"]
    })


reversed_df = cleaned_df.apply(reverse_translation, axis=1)

In [49]:
# Add the reversed translation pairs to the original combined dataset.
cleaned_df = pd.concat(
    [
        cleaned_df,
        reversed_df
    ],
    ignore_index=True
)

In [ ]:
# Confirm the new total size and direction balance after reversal.
print("Original + reversed:", len(cleaned_df))

print(cleaned_df["direction"].value_counts())

In [ ]:
# Inspect ALL duplicate translation pairs
duplicates = cleaned_df[
    cleaned_df.duplicated(
        subset=["source", "target"],
        keep=False
    )
].sort_values(["source", "target"])

print("Duplicate rows:", len(duplicates))
print("Unique duplicate pairs:",
      duplicates[["source", "target"]].drop_duplicates().shape[0])

display(
    duplicates[
        ["source", "target", "direction", "dataset"]
    ]
)

In [ ]:
print("Before:", len(cleaned_df))

cleaned1_df = cleaned_df.drop_duplicates(
    subset=["source", "target"],
    keep="first"
).reset_index(drop=True)

print("After:", len(cleaned1_df))
print("Removed:", len(cleaned_df) - len(cleaned1_df))

In [ ]:
print(cleaned1_df["dataset"].value_counts())
print(cleaned1_df["direction"].value_counts())

In [ ]:
print(len(cleaned1_df))

In [ ]:
cleaned1_df[["source", "target", "dataset", "direction"]]

In [ ]:
final_df = cleaned1_df[
    ["source", "target", "direction"]
].copy()
print("Final dataset shape:", final_df.shape)

print("\nDirection distribution:")
print(final_df["direction"].value_counts())

display(final_df.head(10))

In [57]:
# Start with the final cleaned dataframe
df = final_df.copy()

# Shuffle rows reproducibly
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Create group key so both translation directions stay together
def make_group_key(row):
    return tuple(sorted([row["source"], row["target"]]))

df["group_key"] = df.apply(make_group_key, axis=1)

print("Total rows:", len(df))
print("Unique groups:", df["group_key"].nunique())

In [ ]:
import numpy as np
# Validation target
TARGET_VAL_ROWS = 10000
SEED = 42

# Number of rows in each translation group
group_sizes = df.groupby("group_key").size()

# Get all unique groups
unique_groups = group_sizes.index.to_numpy()

# Randomize the groups
rng = np.random.default_rng(SEED)
shuffled_groups = rng.permutation(unique_groups)

# Select whole groups for validation
val_groups = set()
running_total = 0

for g in shuffled_groups:
    if running_total >= TARGET_VAL_ROWS:
        break

    val_groups.add(g)
    running_total += group_sizes[g]

print("Selected groups:", len(val_groups))
print("Validation rows:", running_total)

In [ ]:
# Identify validation rows
is_val = df["group_key"].isin(val_groups)

# Create train and validation DataFrames
val_df = df[is_val].copy()
train_df = df[~is_val].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

In [ ]:
display(val_df.head(20))
print(val_df["direction"].value_counts())

In [ ]:
# Verify that no translation group exists in both splits
train_groups = set(train_df["group_key"])
val_groups_check = set(val_df["group_key"])

overlap = train_groups & val_groups_check

print("Overlapping groups:", len(overlap))

In [64]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

final_split_dataset = DatasetDict({
    "train": train_dataset,
    "validation": eval_dataset
})

In [ ]:
from huggingface_hub import login

login()

In [ ]:
REPO_ID = "sjcMLT/nep-eng-parallel-corpus"
final_split_dataset.push_to_hub(
    REPO_ID,
    private=True)

In [ ]:
from datasets import load_dataset

check = load_dataset("sjcMLT/nep-eng-parallel-corpus")
print(check)
print(check["train"][0])
print(check["validation"][0])
